In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS 03_global_tech_gold.03_data_cube;

In [0]:
from pyspark.sql.functions import col, lit, coalesce, when, concat_ws

In [0]:
# Load Dimension Tables
dim_date = spark.table("03_global_tech_gold.01_dims_tables.dim_date")
dim_company = spark.table("03_global_tech_gold.01_dims_tables.dim_company")
dim_department = spark.table("03_global_tech_gold.01_dims_tables.dim_department")
dim_employee = spark.table("03_global_tech_gold.01_dims_tables.dim_employee")
dim_account = spark.table("03_global_tech_gold.01_dims_tables.dim_account")

In [0]:
# Load Fact Tables
fact_general_ledger = spark.table("03_global_tech_gold.02_facts_tables.fact_general_ledger")
fact_payroll = spark.table("03_global_tech_gold.02_facts_tables.fact_payroll")

In [0]:
# Join both fact tables on common dimensions
data_cube = fact_general_ledger.alias("gl").join(
    fact_payroll.alias("pr"),
    on=[
        col("gl.company_key") == col("pr.company_key"),
        col("gl.department_key") == col("pr.department_key"),
        col("gl.year_month") == col("pr.year_month")
    ],
    how="full_outer"
).select(
    # Common dimension keys (coalesce to handle nulls from full outer join)
    coalesce(col("gl.company_key"), col("pr.company_key")).alias("company_key"),
    coalesce(col("gl.department_key"), col("pr.department_key")).alias("department_key"),
    coalesce(col("gl.company_id"), col("pr.company_id")).alias("company_id"),
    coalesce(col("gl.department_id"), col("pr.department_id")).alias("department_id"),
    coalesce(col("gl.year_month"), col("pr.year_month")).alias("year_month"),
    
    # General Ledger specific columns (prefixed with gl_)
    col("gl.gl_key"),
    col("gl.entry_date_key").alias("gl_entry_date_key"),
    col("gl.posting_date_key").alias("gl_posting_date_key"),
    col("gl.account_key"),
    col("gl.gl_id"),
    col("gl.account_id"),
    col("gl.entry_date").alias("gl_entry_date"),
    col("gl.posting_date").alias("gl_posting_date"),
    col("gl.fiscal_year"),
    col("gl.fiscal_period"),
    col("gl.transaction_type"),
    col("gl.reference_number").alias("gl_reference_number"),
    col("gl.description").alias("gl_description"),
    col("gl.debit_amount"),
    col("gl.credit_amount"),
    col("gl.created_by"),
    col("gl.approved_by"),
    col("gl.gl_status"),
    col("gl.account_category"),
    
    # Payroll specific columns (prefixed with payroll_)
    col("pr.payroll_key"),
    col("pr.pay_date_key").alias("payroll_pay_date_key"),
    col("pr.pay_period_start_date_key"),
    col("pr.pay_period_end_date_key"),
    col("pr.employee_key"),
    col("pr.payroll_id"),
    col("pr.employee_id"),
    col("pr.bonus"),
    col("pr.overtime_pay"),
    col("pr.commission"),
    col("pr.allowances"),
    col("pr.total_compensation"),
    col("pr.tax_deduction"),
    col("pr.social_security"),
    col("pr.health_insurance"),
    col("pr.retirement_contribution"),
    col("pr.other_deductions"),
    col("pr.total_deduction"),
    col("pr.net_salary"),
    col("pr.payment_method"),
    col("pr.payroll_status")
)

# Display sample
display(data_cube.limit(10))

In [0]:
# Save the unified data cube to the gold layer
data_cube.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("03_global_tech_gold.03_data_cube.unified_data_cube")

print("✓ Unified data cube successfully saved to 03_global_tech_gold.03_data_cube.unified_data_cube")

In [0]:
# Verify the saved data cube
verify_cube = spark.table("03_global_tech_gold.03_data_cube.unified_data_cube")

print(f"Total rows in unified data cube: {verify_cube.count():,}")
print(f"Total columns: {len(verify_cube.columns)}")
print(f"\nColumn names:")
for col_name in verify_cube.columns:
    print(f"  - {col_name}")

# Show sample
print(f"\nSample data:")
display(verify_cube.limit(5))

In [0]:
display(data_cube)